# 07 — GenAI Chatbot: RAG-Based AI Adoption Advisor for SMEs

**Goal:** Build a Retrieval-Augmented Generation (RAG) chatbot that acts as an AI adoption advisor for small and medium-sized enterprises. The advisor retrieves relevant context from a curated knowledge base and generates personalized recommendations.

**Architecture:**
```
User Query (Company Profile)
        ↓
  Retrieval Layer
  (Knowledge Base Search)
        ↓
  Context Injection
  (Top-K relevant entries)
        ↓
  LLM Generation
  (Claude / GPT via API)
        ↓
  Structured Response
  (Score + Recommendations)
```

---

## 1. RAG Architecture Overview

### What is RAG?

**Retrieval-Augmented Generation (RAG)** is a pattern that enhances language model outputs by grounding them in a relevant, domain-specific knowledge base before generating a response.

### Why RAG for SME AI Advisory?

| Problem | RAG Solution |
|---|---|
| General LLMs give generic advice | We retrieve industry + size specific KB entries |
| LLMs may hallucinate ROI figures | Grounded in real adoption rate data from our dataset |
| One-size-fits-all recommendations | Context-aware retrieval filters by company profile |
| LLM knowledge cutoff | KB is updated with our 2019-2024 research findings |

### Pipeline Steps

1. **User submits** company profile: industry, size, employees, revenue, current tools
2. **Retrieval function** scans the knowledge base and returns top-K relevant entries
3. **Prompt builder** injects KB context + company profile into the system prompt
4. **LLM generates** a structured advisory response with score + recommendations
5. **Output formatter** displays the response in a readable format

## 2. Setup & Imports

In [1]:
import os
import json
import re
import math
from pprint import pprint
from IPython.display import display, Markdown, HTML

# Try to import Anthropic SDK
try:
    import anthropic
    ANTHROPIC_AVAILABLE = True
    print('Anthropic SDK available.')
except ImportError:
    ANTHROPIC_AVAILABLE = False
    print('Anthropic SDK not found.')
    print('Install with: pip install anthropic')

# Try to import LangChain components (optional)
try:
    from langchain.schema import HumanMessage, SystemMessage
    from langchain.prompts import ChatPromptTemplate
    LANGCHAIN_AVAILABLE = True
    print('LangChain available.')
except ImportError:
    LANGCHAIN_AVAILABLE = False
    print('LangChain not found (optional). Install with: pip install langchain langchain-anthropic')

print('\nSetup complete. RAG demo will run without API keys using mock responses.')

Anthropic SDK not found.
Install with: pip install anthropic
LangChain not found (optional). Install with: pip install langchain langchain-anthropic

Setup complete. RAG demo will run without API keys using mock responses.


## 3. Knowledge Base

In [2]:
KNOWLEDGE_BASE = [
    # --- Technology & Software ---
    {
        "id": "kb_001",
        "industry": "Technology & Software",
        "size": "Small",
        "employees_range": [10, 100],
        "ai_tools": ["GitHub Copilot", "ChatGPT API", "Hugging Face", "Tabnine"],
        "roi_months": 6,
        "adoption_rate_2024": "97%",
        "top_barrier": "Data quality",
        "avg_investment_usd": 45000,
        "productivity_gain_pct": 28,
        "recommendation": (
            "Start with code generation tools (GitHub Copilot), then automate customer support "
            "with Claude API. Use Hugging Face for custom NLP models on internal data. "
            "Expected ROI: 6-9 months."
        )
    },
    {
        "id": "kb_002",
        "industry": "Technology & Software",
        "size": "Medium",
        "employees_range": [100, 500],
        "ai_tools": ["GitHub Copilot", "OpenAI Enterprise", "DataRobot", "AWS SageMaker"],
        "roi_months": 9,
        "adoption_rate_2024": "94%",
        "top_barrier": "Integration complexity",
        "avg_investment_usd": 120000,
        "productivity_gain_pct": 35,
        "recommendation": (
            "Invest in MLOps infrastructure (AWS SageMaker or Azure ML). "
            "Implement AI-driven QA testing and customer analytics. "
            "Consider hiring an AI/ML engineer. Prioritize data governance first."
        )
    },
    # --- Healthcare & Life Sciences ---
    {
        "id": "kb_003",
        "industry": "Healthcare & Life Sciences",
        "size": "Small",
        "employees_range": [5, 50],
        "ai_tools": ["Epic AI", "Nuance DAX", "Google Health AI"],
        "roi_months": 18,
        "adoption_rate_2024": "72%",
        "top_barrier": "HIPAA compliance & data privacy",
        "avg_investment_usd": 38000,
        "productivity_gain_pct": 22,
        "recommendation": (
            "Begin with clinical documentation AI (Nuance DAX for ambient documentation). "
            "Ensure HIPAA BAA agreements with all AI vendors. "
            "Avoid storing PHI in general-purpose AI tools. ROI comes from admin time savings."
        )
    },
    {
        "id": "kb_004",
        "industry": "Healthcare & Life Sciences",
        "size": "Medium",
        "employees_range": [50, 300],
        "ai_tools": ["Microsoft Azure Health Bot", "IBM Watson Health", "Veeva AI"],
        "roi_months": 24,
        "adoption_rate_2024": "68%",
        "top_barrier": "Regulatory approval & liability",
        "avg_investment_usd": 95000,
        "productivity_gain_pct": 18,
        "recommendation": (
            "Focus on operational AI (scheduling, billing optimization, supply chain). "
            "Clinical AI requires FDA clearance pathway planning. "
            "Partner with healthcare AI vendors who handle compliance. Budget 24-month ROI horizon."
        )
    },
    # --- Manufacturing ---
    {
        "id": "kb_005",
        "industry": "Manufacturing",
        "size": "Small",
        "employees_range": [20, 150],
        "ai_tools": ["Sight Machine", "Uptake", "Microsoft Azure IoT"],
        "roi_months": 15,
        "adoption_rate_2024": "58%",
        "top_barrier": "Legacy equipment integration",
        "avg_investment_usd": 62000,
        "productivity_gain_pct": 19,
        "recommendation": (
            "Start with predictive maintenance AI on your highest-value equipment. "
            "IoT sensors + AI can reduce unplanned downtime by 20-40%. "
            "NIST MEP program offers subsidized AI consulting for small manufacturers."
        )
    },
    {
        "id": "kb_006",
        "industry": "Manufacturing",
        "size": "Medium",
        "employees_range": [150, 500],
        "ai_tools": ["Rockwell FactoryTalk", "GE Predix", "PTC ThingWorx"],
        "roi_months": 18,
        "adoption_rate_2024": "63%",
        "top_barrier": "Workforce upskilling",
        "avg_investment_usd": 185000,
        "productivity_gain_pct": 24,
        "recommendation": (
            "Implement AI-driven demand forecasting and inventory optimization. "
            "Quality inspection AI (computer vision) can reduce defect rates by 30%+. "
            "Run a 90-day pilot on one production line before full rollout."
        )
    },
    # --- Retail & E-commerce ---
    {
        "id": "kb_007",
        "industry": "Retail",
        "size": "Micro",
        "employees_range": [1, 10],
        "ai_tools": ["Shopify AI", "ChatGPT for content", "Jasper AI"],
        "roi_months": 4,
        "adoption_rate_2024": "41%",
        "top_barrier": "Cost & ROI uncertainty",
        "avg_investment_usd": 3200,
        "productivity_gain_pct": 14,
        "recommendation": (
            "Use Shopify AI for product descriptions and SEO optimization. "
            "ChatGPT can handle customer FAQs and email drafts. "
            "Start with free/freemium tools. Budget $200-500/month. ROI in under 6 months."
        )
    },
    {
        "id": "kb_008",
        "industry": "Retail",
        "size": "Small",
        "employees_range": [10, 100],
        "ai_tools": ["Dynamic Yield", "Bloomreach", "Yotpo AI"],
        "roi_months": 10,
        "adoption_rate_2024": "52%",
        "top_barrier": "Data fragmentation",
        "avg_investment_usd": 28000,
        "productivity_gain_pct": 21,
        "recommendation": (
            "Personalization AI drives 15-25% revenue uplift in e-commerce. "
            "Consolidate customer data into a single CDP before adding AI. "
            "AI-powered demand forecasting reduces inventory costs by 10-20%."
        )
    },
    # --- Financial Services ---
    {
        "id": "kb_009",
        "industry": "Financial Services",
        "size": "Small",
        "employees_range": [5, 80],
        "ai_tools": ["Kabbage AI", "QuickBooks AI", "Zest AI"],
        "roi_months": 12,
        "adoption_rate_2024": "81%",
        "top_barrier": "Regulatory compliance (SOC2, SOX)",
        "avg_investment_usd": 55000,
        "productivity_gain_pct": 30,
        "recommendation": (
            "AI fraud detection can reduce losses by 40-60%. "
            "Automated KYC/AML screening dramatically cuts compliance costs. "
            "Use explainable AI models — regulators require decision transparency."
        )
    },
    # --- Professional Services ---
    {
        "id": "kb_010",
        "industry": "Professional Services",
        "size": "Small",
        "employees_range": [5, 100],
        "ai_tools": ["Harvey AI", "Clio AI", "ChatGPT Enterprise", "Otter.ai"],
        "roi_months": 8,
        "adoption_rate_2024": "76%",
        "top_barrier": "Client confidentiality concerns",
        "avg_investment_usd": 32000,
        "productivity_gain_pct": 33,
        "recommendation": (
            "Document automation and contract review AI yield fast ROI for law/consulting firms. "
            "Meeting transcription (Otter.ai) saves 2-3 hours per employee per week. "
            "Use private/on-premises AI deployments to protect client confidentiality."
        )
    },
    # --- Food & Agriculture ---
    {
        "id": "kb_011",
        "industry": "Agriculture",
        "size": "Small",
        "employees_range": [5, 75],
        "ai_tools": ["John Deere Operations Center AI", "Trimble Ag", "Climate.ai"],
        "roi_months": 24,
        "adoption_rate_2024": "29%",
        "top_barrier": "Connectivity & digital literacy",
        "avg_investment_usd": 18000,
        "productivity_gain_pct": 12,
        "recommendation": (
            "Precision agriculture AI reduces input costs (fertilizer, water) by 15-25%. "
            "Start with crop monitoring via satellite imagery AI. "
            "USDA NRCS and Farm Service Agency offer grants for precision ag technology."
        )
    },
    # --- Education ---
    {
        "id": "kb_012",
        "industry": "Education",
        "size": "Small",
        "employees_range": [10, 150],
        "ai_tools": ["Khanmigo", "Turnitin AI", "Grammarly Business", "ChatGPT Edu"],
        "roi_months": 14,
        "adoption_rate_2024": "61%",
        "top_barrier": "Academic integrity concerns",
        "avg_investment_usd": 22000,
        "productivity_gain_pct": 18,
        "recommendation": (
            "AI tutoring tools improve student outcomes while reducing instructor workload. "
            "Implement AI detection policy alongside AI assistance tools. "
            "Administrative AI (scheduling, grading support) has fastest ROI."
        )
    },
    # --- Construction ---
    {
        "id": "kb_013",
        "industry": "Construction",
        "size": "Small",
        "employees_range": [10, 100],
        "ai_tools": ["Procore AI", "PlanGrid AI", "Autodesk Construction Cloud"],
        "roi_months": 20,
        "adoption_rate_2024": "44%",
        "top_barrier": "Workforce age & digital literacy",
        "avg_investment_usd": 35000,
        "productivity_gain_pct": 16,
        "recommendation": (
            "AI-powered project management reduces overruns by 10-20%. "
            "Computer vision safety monitoring cuts workplace incidents. "
            "Start with estimating AI — fastest adoption and clearest ROI."
        )
    },
    # --- Hospitality ---
    {
        "id": "kb_014",
        "industry": "Hospitality",
        "size": "Small",
        "employees_range": [10, 100],
        "ai_tools": ["Revinate AI", "Apaleo", "ChatGPT for guest services"],
        "roi_months": 11,
        "adoption_rate_2024": "48%",
        "top_barrier": "High staff turnover limiting training",
        "avg_investment_usd": 24000,
        "productivity_gain_pct": 20,
        "recommendation": (
            "Dynamic pricing AI boosts RevPAR by 8-15% in hospitality. "
            "AI chatbots handle 70% of guest queries, reducing front desk load. "
            "Sentiment analysis on reviews enables faster reputation management."
        )
    },
    # --- Logistics ---
    {
        "id": "kb_015",
        "industry": "Logistics & Transportation",
        "size": "Medium",
        "employees_range": [50, 400],
        "ai_tools": ["FourKites", "project44", "Locus Robotics"],
        "roi_months": 14,
        "adoption_rate_2024": "69%",
        "top_barrier": "Integration with legacy TMS systems",
        "avg_investment_usd": 110000,
        "productivity_gain_pct": 26,
        "recommendation": (
            "Route optimization AI reduces fuel costs by 10-15%. "
            "Predictive ETAs improve customer satisfaction scores significantly. "
            "Warehouse AI (pick-path optimization) yields 25-40% throughput gains."
        )
    }
]

print(f'Knowledge Base loaded: {len(KNOWLEDGE_BASE)} entries')
industries_in_kb = sorted(set(e['industry'] for e in KNOWLEDGE_BASE))
print(f'Industries covered: {industries_in_kb}')

Knowledge Base loaded: 15 entries
Industries covered: ['Agriculture', 'Construction', 'Education', 'Financial Services', 'Healthcare & Life Sciences', 'Hospitality', 'Logistics & Transportation', 'Manufacturing', 'Professional Services', 'Retail', 'Technology & Software']


## 4. Retrieval Function

In [3]:
def retrieve_context(company_profile: dict, top_k: int = 3) -> list:
    """
    Retrieve the most relevant knowledge base entries for a company profile.
    
    Scoring logic:
      - Industry match:     +10 points
      - Partial industry match (keyword): +5 points
      - Size category match: +5 points
      - Employee range match: +3 points
    
    Args:
        company_profile: dict with keys: industry, size_category, employees, revenue_m_usd
        top_k: number of KB entries to return
    
    Returns:
        List of top-k KB entries sorted by relevance score
    """
    profile_industry = str(company_profile.get('industry', '')).lower()
    profile_size     = str(company_profile.get('size_category', '')).lower()
    profile_emp      = int(company_profile.get('employees', 0))

    scored = []
    for entry in KNOWLEDGE_BASE:
        score = 0

        # Industry match (exact or keyword)
        kb_industry = entry['industry'].lower()
        if profile_industry == kb_industry:
            score += 10
        elif any(word in kb_industry for word in profile_industry.split()) or \
             any(word in profile_industry for word in kb_industry.split()):
            score += 5

        # Size match
        kb_size = entry['size'].lower()
        if profile_size == kb_size:
            score += 5
        elif ('small' in profile_size and kb_size in ['small', 'micro']) or \
             ('micro' in profile_size and kb_size in ['small', 'micro']):
            score += 2

        # Employee range match
        emp_low, emp_high = entry.get('employees_range', [0, 9999])
        if emp_low <= profile_emp <= emp_high:
            score += 3

        scored.append((score, entry))

    # Sort by score descending, take top_k
    scored.sort(key=lambda x: x[0], reverse=True)
    top_entries = [entry for _, entry in scored[:top_k]]

    return top_entries


# Quick test
test_profile = {'industry': 'Technology & Software', 'size_category': 'Small', 'employees': 25}
results = retrieve_context(test_profile, top_k=3)
print(f'Retrieval test for: {test_profile}')
for i, r in enumerate(results, 1):
    print(f'  {i}. [{r["id"]}] {r["industry"]} / {r["size"]} — ROI: {r["roi_months"]} months')

Retrieval test for: {'industry': 'Technology & Software', 'size_category': 'Small', 'employees': 25}
  1. [kb_001] Technology & Software / Small — ROI: 6 months
  2. [kb_003] Healthcare & Life Sciences / Small — ROI: 18 months
  3. [kb_002] Technology & Software / Medium — ROI: 9 months


## 5. Prompt Engineering

In [4]:
SYSTEM_PROMPT = """\
You are an expert AI Adoption Advisor for US small and medium-sized enterprises (SMEs).

Your role is to:
1. Analyze a company's profile and AI readiness
2. Provide concrete, actionable AI adoption recommendations
3. Reference specific tools, expected ROI, and implementation timelines
4. Be realistic about barriers and risks
5. Prioritize quick wins alongside strategic long-term moves

TONE: Professional, practical, encouraging. Avoid jargon. Focus on business outcomes.

FORMAT your response as:
## AI Readiness Assessment
- Overall Score: X/100
- Strengths: ...
- Gaps: ...

## Top 3 Recommended AI Tools
1. [Tool Name] — [Why / Expected benefit]
2. ...
3. ...

## 90-Day Action Plan
- Week 1-4: ...
- Week 5-8: ...
- Week 9-12: ...

## Expected ROI
- Investment: $X - $Y
- Payback period: X months
- Productivity gain: X%

## Risks & Mitigation
- ...
"""


def build_user_prompt(company_profile: dict, kb_context: list) -> str:
    """
    Build the user-turn prompt by injecting company profile + retrieved KB context.
    
    Args:
        company_profile: dict with company details
        kb_context: list of retrieved KB entries
    
    Returns:
        Formatted user prompt string
    """
    context_str = '\n\n'.join([
        f"""[KB Entry {i+1}: {e['id']}]\n"""
        f"""Industry: {e['industry']} | Size: {e['size']} | Adoption Rate 2024: {e['adoption_rate_2024']}\n"""
        f"""Recommended Tools: {', '.join(e['ai_tools'][:3])}\n"""
        f"""Typical Investment: ${e['avg_investment_usd']:,} | ROI Timeline: {e['roi_months']} months\n"""
        f"""Top Barrier: {e['top_barrier']}\n"""
        f"""Advice: {e['recommendation']}"""
        for i, e in enumerate(kb_context)
    ])

    prompt = f"""\
=== RETRIEVED KNOWLEDGE BASE CONTEXT ===
{context_str}

=== COMPANY PROFILE ===
Company: {company_profile.get('name', 'SME Client')}
Industry: {company_profile.get('industry', 'Unknown')}
Size Category: {company_profile.get('size_category', 'Unknown')}
Employees: {company_profile.get('employees', 'Unknown')}
Annual Revenue: ${company_profile.get('revenue_m_usd', 'Unknown')}M
Years in Business: {company_profile.get('years_in_business', 'Unknown')}
Cloud Adopted: {'Yes' if company_profile.get('cloud_adopted') else 'No'}
Has Data Strategy: {'Yes' if company_profile.get('has_data_strategy') else 'No'}
CEO Tech Background: {'Yes' if company_profile.get('ceo_tech_background') else 'No'}
Digital Maturity Score: {company_profile.get('digital_maturity', 'Unknown')}/5
Current AI Tools Count: {company_profile.get('ai_tools_count', 0)}
Primary Barrier: {company_profile.get('primary_barrier', 'Unknown')}
Current AI Situation: {company_profile.get('description', '')}

=== TASK ===
Based on the KB context and company profile above, provide a comprehensive AI adoption advisory.
Prioritize practical recommendations that match this company's size, budget, and industry.
"""
    return prompt


print('Prompt templates defined.')
print(f'System prompt length: {len(SYSTEM_PROMPT)} chars')

Prompt templates defined.
System prompt length: 858 chars


## 6. AI Readiness Score Calculator

In [5]:
def calculate_ai_readiness_score(profile: dict) -> dict:
    """
    Calculate an AI readiness score (0-100) for an SME based on its profile.
    
    Scoring dimensions:
      - Digital Infrastructure (25 pts): cloud adoption, digital maturity
      - Data Readiness       (20 pts): data strategy, data quality indicators
      - Technology Investment (20 pts): tech_invest_pct, tech_emp_pct
      - Leadership & Culture  (15 pts): CEO tech background, openness to change
      - Financial Capacity    (10 pts): revenue, growth
      - Existing AI Use       (10 pts): ai_tools_count, ai_investment
    
    Args:
        profile: dict with company characteristics
    
    Returns:
        dict with overall score, dimension scores, and tier label
    """
    scores = {}

    # 1. Digital Infrastructure (25 pts)
    cloud_pts    = 15 if profile.get('cloud_adopted', False) else 0
    maturity_raw = profile.get('digital_maturity', 1)
    maturity_pts = min(10, int((maturity_raw / 5) * 10))
    scores['digital_infrastructure'] = cloud_pts + maturity_pts

    # 2. Data Readiness (20 pts)
    data_strat_pts = 12 if profile.get('has_data_strategy', False) else 0
    emp_pct        = profile.get('tech_emp_pct', 0)
    tech_emp_pts   = min(8, int((emp_pct / 20) * 8))  # cap at 20% tech employees
    scores['data_readiness'] = data_strat_pts + tech_emp_pts

    # 3. Technology Investment (20 pts)
    invest_pct      = profile.get('tech_invest_pct', 0)
    invest_pts      = min(20, int((invest_pct / 15) * 20))  # cap at 15% tech investment
    scores['tech_investment'] = invest_pts

    # 4. Leadership & Culture (15 pts)
    ceo_pts    = 10 if profile.get('ceo_tech_background', False) else 3
    yib        = profile.get('years_in_business', 10)
    # Newer companies tend to be more digitally native
    age_pts    = max(0, 5 - int(yib / 10))
    scores['leadership_culture'] = ceo_pts + age_pts

    # 5. Financial Capacity (10 pts)
    revenue = profile.get('revenue_m_usd', 0)
    if revenue >= 10:
        fin_pts = 10
    elif revenue >= 5:
        fin_pts = 7
    elif revenue >= 1:
        fin_pts = 4
    else:
        fin_pts = 1
    scores['financial_capacity'] = fin_pts

    # 6. Existing AI Use (10 pts)
    ai_count      = profile.get('ai_tools_count', 0)
    ai_tools_pts  = min(10, ai_count * 2)
    scores['existing_ai_use'] = ai_tools_pts

    # Overall score
    total = sum(scores.values())
    total = min(100, total)

    # Tier classification
    if total >= 80:
        tier = 'AI Pioneer'
        tier_desc = 'Ready for advanced AI integration. Focus on scaling and optimization.'
    elif total >= 60:
        tier = 'Digital Explorer'
        tier_desc = 'Good foundation. Time to systematize AI adoption across functions.'
    elif total >= 40:
        tier = 'Cost-Conscious Adopter'
        tier_desc = 'Selective AI use possible. Prioritize high-ROI, low-cost tools.'
    else:
        tier = 'Traditional SME'
        tier_desc = 'Foundation work needed first. Focus on cloud and data infrastructure.'

    return {
        'overall_score': total,
        'tier': tier,
        'tier_description': tier_desc,
        'dimension_scores': scores,
        'max_possible': {
            'digital_infrastructure': 25,
            'data_readiness': 20,
            'tech_investment': 20,
            'leadership_culture': 15,
            'financial_capacity': 10,
            'existing_ai_use': 10
        }
    }


def display_readiness_score(result: dict, company_name: str = 'SME'):
    """Pretty-print the AI readiness score."""
    score   = result['overall_score']
    tier    = result['tier']
    desc    = result['tier_description']
    dims    = result['dimension_scores']
    maxvals = result['max_possible']

    bar_len  = 30
    filled   = int((score / 100) * bar_len)
    bar      = '█' * filled + '░' * (bar_len - filled)

    print(f'\n{"=" * 55}')
    print(f'  AI READINESS SCORE — {company_name}')
    print(f'{"=" * 55}')
    print(f'  Overall:  {score}/100  [{bar}]')
    print(f'  Tier:     {tier}')
    print(f'  Summary:  {desc}')
    print(f'  {"─" * 51}')
    print(f'  Dimension Breakdown:')
    for dim, val in dims.items():
        max_val  = maxvals[dim]
        dim_bar  = '█' * int((val / max_val) * 15) + '░' * (15 - int((val / max_val) * 15))
        label    = dim.replace('_', ' ').title()
        print(f'    {label:<26} {val:2d}/{max_val}  [{dim_bar}]')
    print(f'{"=" * 55}\n')


# Quick test
test_score = calculate_ai_readiness_score({
    'cloud_adopted': True, 'digital_maturity': 3.5, 'has_data_strategy': True,
    'tech_emp_pct': 12, 'tech_invest_pct': 8, 'ceo_tech_background': True,
    'years_in_business': 7, 'revenue_m_usd': 6, 'ai_tools_count': 3
})
display_readiness_score(test_score, 'Test Company')


  AI READINESS SCORE — Test Company
  Overall:  76/100  [██████████████████████░░░░░░░░]
  Tier:     Digital Explorer
  Summary:  Good foundation. Time to systematize AI adoption across functions.
  ───────────────────────────────────────────────────
  Dimension Breakdown:
    Digital Infrastructure     22/25  [█████████████░░]
    Data Readiness             16/20  [████████████░░░]
    Tech Investment            10/20  [███████░░░░░░░░]
    Leadership Culture         15/15  [███████████████]
    Financial Capacity          7/10  [██████████░░░░░]
    Existing Ai Use             6/10  [█████████░░░░░░]



## 7. Demo Without API — Three Company Profiles

In [6]:
# ======================================================
# Define three demo company profiles
# ======================================================

PROFILE_A = {
    "name": "Quantum Leaf Technologies",
    "industry": "Technology & Software",
    "size_category": "Small",
    "employees": 15,
    "revenue_m_usd": 2.0,
    "years_in_business": 4,
    "cloud_adopted": True,
    "has_data_strategy": True,
    "ceo_tech_background": True,
    "digital_maturity": 4.0,
    "tech_invest_pct": 12.0,
    "tech_emp_pct": 40.0,
    "ai_tools_count": 2,
    "primary_barrier": "Data quality",
    "description": (
        "B2B SaaS startup building project management tools. Already using GitHub Copilot. "
        "Want to add AI features to product and automate customer support."
    )
}

PROFILE_B = {
    "name": "Great Lakes Precision Parts",
    "industry": "Manufacturing",
    "size_category": "Medium",
    "employees": 180,
    "revenue_m_usd": 25.0,
    "years_in_business": 32,
    "cloud_adopted": False,
    "has_data_strategy": False,
    "ceo_tech_background": False,
    "digital_maturity": 2.0,
    "tech_invest_pct": 2.5,
    "tech_emp_pct": 5.0,
    "ai_tools_count": 0,
    "primary_barrier": "Legacy systems & cost",
    "description": (
        "CNC machining shop serving aerospace and automotive clients. "
        "On-premise ERP system from 2008. Owner wants to explore AI for "
        "predictive maintenance and quality control."
    )
}

PROFILE_C = {
    "name": "Sunflower Boutique",
    "industry": "Retail",
    "size_category": "Micro",
    "employees": 5,
    "revenue_m_usd": 0.3,
    "years_in_business": 8,
    "cloud_adopted": True,
    "has_data_strategy": False,
    "ceo_tech_background": False,
    "digital_maturity": 2.5,
    "tech_invest_pct": 1.5,
    "tech_emp_pct": 0.0,
    "ai_tools_count": 0,
    "primary_barrier": "Budget & awareness",
    "description": (
        "Independent women's fashion boutique in Austin, TX. "
        "Uses Shopify and Instagram. Owner wants to use AI to write product "
        "descriptions and manage social media posts more efficiently."
    )
}

DEMO_PROFILES = [PROFILE_A, PROFILE_B, PROFILE_C]
print(f'Defined {len(DEMO_PROFILES)} demo company profiles.')
for p in DEMO_PROFILES:
    print(f"  - {p['name']} ({p['industry']}, {p['employees']} employees, ${p['revenue_m_usd']}M revenue)")

Defined 3 demo company profiles.
  - Quantum Leaf Technologies (Technology & Software, 15 employees, $2.0M revenue)
  - Great Lakes Precision Parts (Manufacturing, 180 employees, $25.0M revenue)
  - Sunflower Boutique (Retail, 5 employees, $0.3M revenue)


In [7]:
# ======================================================
# Mock advisory responses (no API key required)
# ======================================================

MOCK_RESPONSES = {
    "Quantum Leaf Technologies": """
## AI Readiness Assessment
- **Overall Score:** 82/100 — Tier: AI Pioneer
- **Strengths:** Cloud-native, high tech investment ratio, tech-savvy CEO, existing AI use
- **Gaps:** Data quality processes, formal ML pipeline, AI governance policy

## Top 3 Recommended AI Tools
1. **Claude API (Anthropic)** — Embed AI-powered features directly into your SaaS product (document analysis, smart suggestions, natural language queries). Estimated integration: 3-4 weeks. Revenue uplift: 15-25% through premium tier.
2. **LangChain + RAG pipeline** — Build a customer support bot trained on your product docs. Reduces support tickets by 40-60%, improves response time from hours to seconds.
3. **Mixpanel AI / PostHog** — AI-powered product analytics to understand user behavior and predict churn. Actionable within days of integration.

## 90-Day Action Plan
- **Week 1-4:** Integrate Claude API into one product feature (e.g., smart search or document summarization). Set up error logging and human review queue.
- **Week 5-8:** Deploy RAG-based customer support chatbot using your existing help center content as the knowledge base.
- **Week 9-12:** Launch AI-enhanced premium tier at +20% price point. Measure conversion and churn impact. Set up A/B testing framework.

## Expected ROI
- **Investment:** $30,000 - $50,000 (dev time + API costs)
- **Payback period:** 6-8 months
- **Productivity gain:** 28-35% (engineering and support teams)
- **Revenue impact:** +$80K-$150K ARR from AI premium features

## Risks & Mitigation
- **Data quality:** Implement data validation pipelines before feeding data to AI models
- **AI hallucination:** Use RAG over internal docs only; add human review for critical outputs
- **API cost overrun:** Set hard spending limits on OpenAI/Anthropic dashboards from day 1
""",

    "Great Lakes Precision Parts": """
## AI Readiness Assessment
- **Overall Score:** 31/100 — Tier: Traditional SME (on the cusp of Cost-Conscious Adopter)
- **Strengths:** Stable revenue ($25M), long business history (32 years), clear use case in mind
- **Gaps:** No cloud infrastructure, no data strategy, no AI tools, low digital maturity

## Top 3 Recommended AI Tools
1. **Azure IoT Hub + Azure Machine Learning** — Start with IoT sensors on 2-3 machines, then apply predictive maintenance AI. Microsoft offers manufacturing-specific starter packages.
2. **Sight Machine** — Purpose-built AI for manufacturing analytics. Connects to existing PLCs and extracts production data without replacing your equipment.
3. **Microsoft 365 Copilot** — Quick win: AI for email, Excel analysis, and meeting summaries. No infrastructure change needed. Gets staff comfortable with AI before bigger rollouts.

## 90-Day Action Plan
- **Week 1-4:** FOUNDATION FIRST — Move email and documents to Microsoft 365 (cloud). Engage NIST MEP (free manufacturing AI consulting). Identify 1 machine for IoT pilot.
- **Week 5-8:** Install IoT sensors on pilot machine. Begin collecting baseline data. Deploy M365 Copilot for office staff.
- **Week 9-12:** Analyze first 30 days of machine data. Get AI vendor demo with your own data. Develop 12-month AI roadmap with CFO buy-in.

## Expected ROI
- **Investment:** $45,000 - $80,000 (Year 1, including infrastructure and consulting)
- **Payback period:** 18-24 months
- **Productivity gain:** 12-18% initially, growing to 25% by Year 3
- **Cost savings:** $200K-$400K/year from reduced downtime once predictive maintenance is running

## Risks & Mitigation
- **Legacy integration:** Hire a systems integrator familiar with manufacturing OT/IT convergence
- **Workforce resistance:** Involve shop floor workers in pilot design; position AI as safety/ease improvement
- **Budget overrun:** Use phased approach — each phase must prove ROI before next investment
""",

    "Sunflower Boutique": """
## AI Readiness Assessment
- **Overall Score:** 24/100 — Tier: Traditional SME
- **Strengths:** Already on Shopify (cloud), clear AI use case identified, long track record
- **Gaps:** Tiny budget, no tech staff, no data strategy, very low AI exposure

## Top 3 Recommended AI Tools
1. **Shopify Magic (built-in, free)** — AI-generated product descriptions with one click. Already included in your Shopify plan. Start here TODAY. Zero additional cost.
2. **ChatGPT (GPT-4o, $20/month)** — Write Instagram captions, email campaigns, blog posts, and customer replies. Creates a week of content in 30 minutes.
3. **Canva AI (free tier)** — AI-powered image generation and design for social media posts. No design skills needed. Keeps your feed looking professional on a micro-budget.

## 90-Day Action Plan
- **Week 1-4:** IMMEDIATE WINS — Enable Shopify Magic for all products. Set up ChatGPT account. Create a simple content calendar template. Spend 2 hours learning prompting basics.
- **Week 5-8:** Systematize content creation: use ChatGPT to batch-create 2 weeks of social content in one sitting. Test AI-written vs. manual product descriptions (Shopify A/B test).
- **Week 9-12:** Review time savings and conversion data. If social engagement improved, consider upgrading to Meta Ads AI targeting ($50-100/month ad budget). Explore AI email marketing (Klaviyo AI).

## Expected ROI
- **Investment:** $20 - $100/month (ChatGPT + optional tools)
- **Payback period:** 1-2 months (immediate time savings)
- **Time savings:** 5-8 hours/week on content creation
- **Revenue impact:** 10-20% uplift from better product descriptions and more consistent social presence

## Risks & Mitigation
- **AI-generated content feels generic:** Always add your personal voice and brand details in the prompt; review before posting
- **Over-reliance on AI:** Keep the authentic boutique personality — customers buy from YOU, not an algorithm
- **Data privacy:** Never enter customer personal information into free AI tools
"""
}

print('Mock responses defined for all 3 profiles.')

Mock responses defined for all 3 profiles.


In [8]:
def run_rag_advisor_demo(profile: dict) -> None:
    """
    Run the full RAG advisor pipeline in demo mode (no API key required).
    Uses mock LLM responses to demonstrate the complete flow.
    """
    name = profile['name']
    print(f'\n{"═" * 60}')
    print(f'  ADVISOR SESSION: {name}')
    print(f'{"═" * 60}')

    # Step 1: Calculate readiness score
    score_result = calculate_ai_readiness_score(profile)
    display_readiness_score(score_result, name)

    # Step 2: Retrieve KB context
    kb_context = retrieve_context(profile, top_k=3)
    print(f'Retrieved {len(kb_context)} KB entries:')
    for e in kb_context:
        print(f'  - [{e["id"]}] {e["industry"]} / {e["size"]}  (adoption 2024: {e["adoption_rate_2024"]})')

    # Step 3: Build prompt (show structure)
    user_prompt = build_user_prompt(profile, kb_context)
    print(f'\nGenerated prompt: {len(user_prompt)} chars')

    # Step 4: Use mock response
    print(f'\n[DEMO MODE: using mock LLM response — no API call made]')
    response_text = MOCK_RESPONSES.get(name, 'No mock response available for this profile.')

    # Display formatted response
    print(f'\n{"─" * 60}')
    print(f'  AI ADOPTION ADVISOR RESPONSE for {name}')
    print(f'{"─" * 60}')
    display(Markdown(response_text))


# Run demo for all three profiles
for profile in DEMO_PROFILES:
    run_rag_advisor_demo(profile)


════════════════════════════════════════════════════════════
  ADVISOR SESSION: Quantum Leaf Technologies
════════════════════════════════════════════════════════════

  AI READINESS SCORE — Quantum Leaf Technologies
  Overall:  82/100  [████████████████████████░░░░░░]
  Tier:     AI Pioneer
  Summary:  Ready for advanced AI integration. Focus on scaling and optimization.
  ───────────────────────────────────────────────────
  Dimension Breakdown:
    Digital Infrastructure     23/25  [█████████████░░]
    Data Readiness             20/20  [███████████████]
    Tech Investment            16/20  [████████████░░░]
    Leadership Culture         15/15  [███████████████]
    Financial Capacity          4/10  [██████░░░░░░░░░]
    Existing Ai Use             4/10  [██████░░░░░░░░░]

Retrieved 3 KB entries:
  - [kb_001] Technology & Software / Small  (adoption 2024: 97%)
  - [kb_003] Healthcare & Life Sciences / Small  (adoption 2024: 72%)
  - [kb_002] Technology & Software / Medium  (adopt


## AI Readiness Assessment
- **Overall Score:** 82/100 — Tier: AI Pioneer
- **Strengths:** Cloud-native, high tech investment ratio, tech-savvy CEO, existing AI use
- **Gaps:** Data quality processes, formal ML pipeline, AI governance policy

## Top 3 Recommended AI Tools
1. **Claude API (Anthropic)** — Embed AI-powered features directly into your SaaS product (document analysis, smart suggestions, natural language queries). Estimated integration: 3-4 weeks. Revenue uplift: 15-25% through premium tier.
2. **LangChain + RAG pipeline** — Build a customer support bot trained on your product docs. Reduces support tickets by 40-60%, improves response time from hours to seconds.
3. **Mixpanel AI / PostHog** — AI-powered product analytics to understand user behavior and predict churn. Actionable within days of integration.

## 90-Day Action Plan
- **Week 1-4:** Integrate Claude API into one product feature (e.g., smart search or document summarization). Set up error logging and human review queue.
- **Week 5-8:** Deploy RAG-based customer support chatbot using your existing help center content as the knowledge base.
- **Week 9-12:** Launch AI-enhanced premium tier at +20% price point. Measure conversion and churn impact. Set up A/B testing framework.

## Expected ROI
- **Investment:** $30,000 - $50,000 (dev time + API costs)
- **Payback period:** 6-8 months
- **Productivity gain:** 28-35% (engineering and support teams)
- **Revenue impact:** +$80K-$150K ARR from AI premium features

## Risks & Mitigation
- **Data quality:** Implement data validation pipelines before feeding data to AI models
- **AI hallucination:** Use RAG over internal docs only; add human review for critical outputs
- **API cost overrun:** Set hard spending limits on OpenAI/Anthropic dashboards from day 1



════════════════════════════════════════════════════════════
  ADVISOR SESSION: Great Lakes Precision Parts
════════════════════════════════════════════════════════════

  AI READINESS SCORE — Great Lakes Precision Parts
  Overall:  24/100  [███████░░░░░░░░░░░░░░░░░░░░░░░]
  Tier:     Traditional SME
  Summary:  Foundation work needed first. Focus on cloud and data infrastructure.
  ───────────────────────────────────────────────────
  Dimension Breakdown:
    Digital Infrastructure      4/25  [██░░░░░░░░░░░░░]
    Data Readiness              2/20  [█░░░░░░░░░░░░░░]
    Tech Investment             3/20  [██░░░░░░░░░░░░░]
    Leadership Culture          5/15  [█████░░░░░░░░░░]
    Financial Capacity         10/10  [███████████████]
    Existing Ai Use             0/10  [░░░░░░░░░░░░░░░]

Retrieved 3 KB entries:
  - [kb_006] Manufacturing / Medium  (adoption 2024: 63%)
  - [kb_005] Manufacturing / Small  (adoption 2024: 58%)
  - [kb_002] Technology & Software / Medium  (adoption 2024: 9


## AI Readiness Assessment
- **Overall Score:** 31/100 — Tier: Traditional SME (on the cusp of Cost-Conscious Adopter)
- **Strengths:** Stable revenue ($25M), long business history (32 years), clear use case in mind
- **Gaps:** No cloud infrastructure, no data strategy, no AI tools, low digital maturity

## Top 3 Recommended AI Tools
1. **Azure IoT Hub + Azure Machine Learning** — Start with IoT sensors on 2-3 machines, then apply predictive maintenance AI. Microsoft offers manufacturing-specific starter packages.
2. **Sight Machine** — Purpose-built AI for manufacturing analytics. Connects to existing PLCs and extracts production data without replacing your equipment.
3. **Microsoft 365 Copilot** — Quick win: AI for email, Excel analysis, and meeting summaries. No infrastructure change needed. Gets staff comfortable with AI before bigger rollouts.

## 90-Day Action Plan
- **Week 1-4:** FOUNDATION FIRST — Move email and documents to Microsoft 365 (cloud). Engage NIST MEP (free manufacturing AI consulting). Identify 1 machine for IoT pilot.
- **Week 5-8:** Install IoT sensors on pilot machine. Begin collecting baseline data. Deploy M365 Copilot for office staff.
- **Week 9-12:** Analyze first 30 days of machine data. Get AI vendor demo with your own data. Develop 12-month AI roadmap with CFO buy-in.

## Expected ROI
- **Investment:** $45,000 - $80,000 (Year 1, including infrastructure and consulting)
- **Payback period:** 18-24 months
- **Productivity gain:** 12-18% initially, growing to 25% by Year 3
- **Cost savings:** $200K-$400K/year from reduced downtime once predictive maintenance is running

## Risks & Mitigation
- **Legacy integration:** Hire a systems integrator familiar with manufacturing OT/IT convergence
- **Workforce resistance:** Involve shop floor workers in pilot design; position AI as safety/ease improvement
- **Budget overrun:** Use phased approach — each phase must prove ROI before next investment



════════════════════════════════════════════════════════════
  ADVISOR SESSION: Sunflower Boutique
════════════════════════════════════════════════════════════

  AI READINESS SCORE — Sunflower Boutique
  Overall:  31/100  [█████████░░░░░░░░░░░░░░░░░░░░░]
  Tier:     Traditional SME
  Summary:  Foundation work needed first. Focus on cloud and data infrastructure.
  ───────────────────────────────────────────────────
  Dimension Breakdown:
    Digital Infrastructure     20/25  [████████████░░░]
    Data Readiness              0/20  [░░░░░░░░░░░░░░░]
    Tech Investment             2/20  [█░░░░░░░░░░░░░░]
    Leadership Culture          8/15  [████████░░░░░░░]
    Financial Capacity          1/10  [█░░░░░░░░░░░░░░]
    Existing Ai Use             0/10  [░░░░░░░░░░░░░░░]

Retrieved 3 KB entries:
  - [kb_007] Retail / Micro  (adoption 2024: 41%)
  - [kb_008] Retail / Small  (adoption 2024: 52%)
  - [kb_003] Healthcare & Life Sciences / Small  (adoption 2024: 72%)

Generated prompt: 2065 c


## AI Readiness Assessment
- **Overall Score:** 24/100 — Tier: Traditional SME
- **Strengths:** Already on Shopify (cloud), clear AI use case identified, long track record
- **Gaps:** Tiny budget, no tech staff, no data strategy, very low AI exposure

## Top 3 Recommended AI Tools
1. **Shopify Magic (built-in, free)** — AI-generated product descriptions with one click. Already included in your Shopify plan. Start here TODAY. Zero additional cost.
2. **ChatGPT (GPT-4o, $20/month)** — Write Instagram captions, email campaigns, blog posts, and customer replies. Creates a week of content in 30 minutes.
3. **Canva AI (free tier)** — AI-powered image generation and design for social media posts. No design skills needed. Keeps your feed looking professional on a micro-budget.

## 90-Day Action Plan
- **Week 1-4:** IMMEDIATE WINS — Enable Shopify Magic for all products. Set up ChatGPT account. Create a simple content calendar template. Spend 2 hours learning prompting basics.
- **Week 5-8:** Systematize content creation: use ChatGPT to batch-create 2 weeks of social content in one sitting. Test AI-written vs. manual product descriptions (Shopify A/B test).
- **Week 9-12:** Review time savings and conversion data. If social engagement improved, consider upgrading to Meta Ads AI targeting ($50-100/month ad budget). Explore AI email marketing (Klaviyo AI).

## Expected ROI
- **Investment:** $20 - $100/month (ChatGPT + optional tools)
- **Payback period:** 1-2 months (immediate time savings)
- **Time savings:** 5-8 hours/week on content creation
- **Revenue impact:** 10-20% uplift from better product descriptions and more consistent social presence

## Risks & Mitigation
- **AI-generated content feels generic:** Always add your personal voice and brand details in the prompt; review before posting
- **Over-reliance on AI:** Keep the authentic boutique personality — customers buy from YOU, not an algorithm
- **Data privacy:** Never enter customer personal information into free AI tools


## 8. Integration with Anthropic API (Optional — Requires API Key)

In [9]:
# ============================================================
# LIVE API INTEGRATION — Requires ANTHROPIC_API_KEY env var
# ============================================================
#
# To enable: set your API key before running:
#   export ANTHROPIC_API_KEY="sk-ant-..."
# Or set it in Python:
#   import os; os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
# ============================================================

def run_rag_advisor_live(profile: dict, model: str = 'claude-opus-4-5') -> str:
    """
    Full live RAG pipeline using the Anthropic API.
    
    Args:
        profile: company profile dict
        model: Anthropic model ID (default: claude-opus-4-5)
    
    Returns:
        Advisory response string from the model
    """
    if not ANTHROPIC_AVAILABLE:
        raise ImportError('Run: pip install anthropic')

    api_key = os.environ.get('ANTHROPIC_API_KEY')
    if not api_key:
        raise ValueError(
            'ANTHROPIC_API_KEY not set. '
            'Set it with: os.environ["ANTHROPIC_API_KEY"] = "your-key"'
        )

    # Initialize client
    client = anthropic.Anthropic(api_key=api_key)

    # Step 1: Retrieve context
    kb_context  = retrieve_context(profile, top_k=3)

    # Step 2: Build prompt
    user_prompt = build_user_prompt(profile, kb_context)

    # Step 3: Call API
    message = client.messages.create(
        model=model,
        max_tokens=2048,
        system=SYSTEM_PROMPT,
        messages=[
            {"role": "user", "content": user_prompt}
        ]
    )

    return message.content[0].text


# --- Demonstrate how to call it (won't execute without valid key) ---
print('=== Live API Usage Example ===')
print('\nCode to run (requires ANTHROPIC_API_KEY):')
print('''
import os
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-your-key-here'

response = run_rag_advisor_live(PROFILE_A)
display(Markdown(response))
''')

# Conditional live call if key is available
if ANTHROPIC_AVAILABLE and os.environ.get('ANTHROPIC_API_KEY'):
    print('\nAPI key detected! Running live advisory for Profile A...')
    try:
        live_response = run_rag_advisor_live(PROFILE_A)
        print(f'\nLive response received ({len(live_response)} chars):')
        display(Markdown(live_response))
    except Exception as e:
        print(f'API call failed: {e}')
else:
    print('\nNo API key found — live section skipped. Mock responses shown above.')

=== Live API Usage Example ===

Code to run (requires ANTHROPIC_API_KEY):

import os
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-your-key-here'

response = run_rag_advisor_live(PROFILE_A)
display(Markdown(response))


No API key found — live section skipped. Mock responses shown above.


## 9. Sample Conversation Output

In [10]:
def display_conversation(turns: list) -> None:
    """
    Display a formatted conversation between SME owner and AI advisor.
    
    Args:
        turns: list of dicts with 'role' ('user' or 'advisor') and 'content'
    """
    html_parts = ['<div style="font-family: Arial, sans-serif; max-width: 800px;">']

    for turn in turns:
        role    = turn['role']
        content = turn['content']

        if role == 'user':
            html_parts.append(f"""
            <div style="background:#E8F4FF; border-left:4px solid #2874A6;
                        margin:10px 0; padding:12px 16px; border-radius:4px;">
                <strong style="color:#2874A6;">SME Owner:</strong>
                <p style="margin:6px 0 0 0; color:#333;">{content}</p>
            </div>
            """)
        else:
            html_parts.append(f"""
            <div style="background:#F0F7E6; border-left:4px solid #6B7C3A;
                        margin:10px 0; padding:12px 16px; border-radius:4px;">
                <strong style="color:#6B7C3A;">AI Adoption Advisor:</strong>
                <p style="margin:6px 0 0 0; color:#333; white-space:pre-wrap;">{content}</p>
            </div>
            """)

    html_parts.append('</div>')
    display(HTML(''.join(html_parts)))


# Sample full conversation
sample_conversation = [
    {
        "role": "user",
        "content": (
            "Hi, I run a boutique fashion shop with 5 employees and about $300K in annual revenue. "
            "I use Shopify and Instagram. I've heard a lot about AI lately and I'm wondering if "
            "it's even relevant for a small shop like mine, or if it's just for big companies."
        )
    },
    {
        "role": "advisor",
        "content": (
            "Great question! AI is absolutely relevant for businesses your size — in fact, "
            "some of the quickest wins happen at the micro-business level because you can "
            "start with free or very low-cost tools and see results within days.\n\n"
            "Based on your profile, here's what I'd recommend starting with today:\n\n"
            "1. Shopify Magic (free, built-in) — One-click AI product descriptions. "
            "You can update all your listings this week.\n"
            "2. ChatGPT ($20/month) — Batch-create your Instagram captions and email campaigns "
            "in one sitting. Most boutique owners save 5-8 hours per week.\n\n"
            "Your AI readiness score is 24/100, which puts you in 'Traditional SME' territory — "
            "but that's not a problem. It just means we start simple and build up.\n\n"
            "Would you like me to walk you through setting up Shopify Magic step by step?"
        )
    },
    {
        "role": "user",
        "content": (
            "That sounds manageable! But I'm worried my customers can tell when content is AI-generated. "
            "Won't it make my boutique feel less personal?"
        )
    },
    {
        "role": "advisor",
        "content": (
            "That's the most common concern — and it's valid. AI-generated content CAN feel generic, "
            "but only if you let it publish without editing.\n\n"
            "Think of AI as your first draft writer, not your voice. The workflow I recommend:\n"
            "  • Give ChatGPT your brand guidelines: 'Write in a warm, feminine, Texas-inspired tone "
            "for a boutique called Sunflower that celebrates individuality'\n"
            "  • AI generates 3 caption options in 10 seconds\n"
            "  • You pick the best one and add your personal touch (1-2 sentences)\n\n"
            "Result: You spend 5 minutes instead of 30, and the final content still sounds like you.\n\n"
            "Many boutique owners actually find AI helps them post MORE consistently, which is "
            "what builds customer connection over time. Quality AND consistency beat authenticity-but-sporadic."
        )
    },
    {
        "role": "user",
        "content": "That makes sense. What's the first thing I should actually do today?"
    },
    {
        "role": "advisor",
        "content": (
            "Today's single action: Enable Shopify Magic.\n\n"
            "1. Log into your Shopify admin\n"
            "2. Go to Products → click any product\n"
            "3. In the Description field, look for the sparkle icon (✨)\n"
            "4. Click it, describe the product briefly, hit Generate\n"
            "5. Edit to add your brand voice, save\n\n"
            "Takes 5 minutes. Do it for your 5 best-selling products this afternoon. "
            "Come back tomorrow and we'll tackle your Instagram workflow. "
            "Small consistent steps are how boutique owners win with AI."
        )
    }
]

print('=== Sample Advisor Conversation ===')
display_conversation(sample_conversation)

=== Sample Advisor Conversation ===


## 10. Chatbot UI Sketch — Streamlit Integration Plan

### How to Build the Streamlit App

The RAG advisor chatbot can be deployed as a Streamlit web application in `app/streamlit_app.py`. Here is the integration plan:

#### File Structure
```
app/
├── streamlit_app.py          ← Main Streamlit UI
├── advisor/
│   ├── knowledge_base.py     ← KNOWLEDGE_BASE dict (from this notebook)
│   ├── retrieval.py          ← retrieve_context() function
│   ├── scoring.py            ← calculate_ai_readiness_score()
│   ├── prompts.py            ← SYSTEM_PROMPT + build_user_prompt()
│   └── api_client.py         ← run_rag_advisor_live() with Anthropic client
```

#### Streamlit UI Components

```python
import streamlit as st
from advisor.scoring import calculate_ai_readiness_score
from advisor.retrieval import retrieve_context
from advisor.api_client import run_rag_advisor_live

st.set_page_config(page_title='SME AI Adoption Advisor', page_icon='🤖', layout='wide')
st.title('AI Adoption Advisor for US SMEs')

# --- Sidebar: Company Profile Form ---
with st.sidebar:
    st.header('Your Company Profile')
    industry       = st.selectbox('Industry', INDUSTRY_OPTIONS)
    size_category  = st.selectbox('Company Size', ['Micro (<10)', 'Small (10-100)', 'Medium (100-500)'])
    employees      = st.slider('Employees', 1, 500, 25)
    revenue        = st.number_input('Annual Revenue ($M)', 0.1, 100.0, 2.0)
    cloud_adopted  = st.checkbox('Using cloud services?')
    data_strategy  = st.checkbox('Have a data strategy?')
    ceo_tech       = st.checkbox('CEO has tech background?')
    digital_score  = st.slider('Digital Maturity (1-5)', 1, 5, 3)
    barrier        = st.selectbox('Primary AI Barrier', BARRIER_OPTIONS)

    if st.button('Get AI Readiness Score', type='primary'):
        profile = dict(industry=industry, size_category=size_category, ...)
        score   = calculate_ai_readiness_score(profile)
        st.metric('Readiness Score', f"{score['overall_score']}/100")
        st.info(f"Tier: {score['tier']}")

# --- Main: Chat Interface ---
st.header('Chat with Your AI Advisor')
if 'messages' not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg['role']):
        st.markdown(msg['content'])

if prompt := st.chat_input('Ask about AI adoption for your business...'):
    st.session_state.messages.append({'role': 'user', 'content': prompt})
    with st.chat_message('assistant'):
        with st.spinner('Retrieving context and generating advice...'):
            response = run_rag_advisor_live(profile, context=prompt)
            st.markdown(response)
    st.session_state.messages.append({'role': 'assistant', 'content': response})
```

#### To Run Locally
```bash
pip install streamlit anthropic
export ANTHROPIC_API_KEY="sk-ant-..."
streamlit run app/streamlit_app.py
```

#### Deployment
- **Streamlit Cloud:** Connect GitHub repo, add `ANTHROPIC_API_KEY` as a secret
- **AWS/GCP/Azure:** Containerize with Docker, use secrets manager for API key
- **Rate limiting:** Add session-based request throttling to manage API costs